In [1]:
import pandas as pd

# Load dataset 
df = pd.read_csv("Matuu_hospital.csv", on_bad_lines="skip")
print(df.head())

# Checking original Columns
print(df.columns) 
df1b = df.drop(
    columns="Unnamed: 0",
    errors="ignore"
)

# Drop duplicates, completely empty rows, and completely empty columns
df1 = (df1b.drop_duplicates()
      .dropna(axis=0, how="all")
      .dropna(axis=1, how="all")
      .reset_index(drop=True)
     )

# Clean column names
df1.columns = (df1.columns
    .str.replace(r"[^a-zA-Z0-9]", " ", regex=True)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
)

# Rename columns
column_updates = {
    "hasinsurance_verify": "has_insurance_verify", 
    "dischargedate": "discharge_date",
    "admissiondate": "admission_date",
    "medicalcondition": "medical_condition", 
    "patientname": "patient_name", 
    "patientid": "patient_id", 
}

df1 = df1.rename(columns=column_updates)

# Sort df by patient ID 
df1 = df1.sort_values(
    by="patient_id", 
    ascending=True
)

# Checking duplicated patient ids
print(df1["patient_name"].duplicated().sum())
print(df1["patient_name"].isna().sum())

# Checking duplicated patient ids
print(df1.loc[df1["patient_id"].duplicated(keep=False)])

# Clean duplicated ids, using keys from patient name column
update_patient_id = {
    "Patient_11": "P111_A", 
    "Patient_20": "P120_A", 
}

df1["patient_id"] = (
    df1["patient_name"].map(update_patient_id)
    .fillna(df1["patient_id"])
)
print(df1.loc[df1["patient_id"].duplicated(keep=False)]) # No duplicates

print(df1["patient_id"].nunique())
print(df1["patient_id"].isna().sum()) # No NaN

# Cleaning medical_condition column
print(df1["medical_condition"].value_counts())

df1["medical_condition"] = (df1["medical_condition"]
                             .str.strip()
                             .str.lower()
                            )

med_updates = {
    "typhoid": "Typhoid", 
    "malaria": "Malaria", 
    "diabetes": "Diabetes", 
    "tb": "TB", 
    "cholera": "Cholera", 
    "hypertension": "Hypertension", 
    "covid-19": "Covid-19", 
    "malariaa": "Malaria", 
    "diabtes": "Diabetes", 
}

df1["medical_condition_clean"] = df1["medical_condition"].map(med_updates)
print(df1["medical_condition_clean"].isna().sum())

df1 = df1.dropna(subset=["medical_condition_clean"])

# Cleaning has_insurance_verify column
print(df1["has_insurance_verify"].value_counts())

update_has_insurance_verify = {
    0: "No", 
    "Yes": "Yes", 
    "TRUE": "Yes", 
    "FALSE": "No", 
    "No": "No", 
    "N": "No", 
    "Y": "Yes", 
    "n": "No", 
    "y": "Yes", 
    "yes": "Yes", 
    1: "Yes", 
    "False": "No", 
    "no": "No", 
    "True": "Yes", 
    "0": "No", 
    "1": "Yes", 
}

df1["has_insurance_verify_clean"] = pd.Categorical(
    df1["has_insurance_verify"].map(update_has_insurance_verify), 
    categories=["Yes", "No"], 
    ordered=False, 
)
print(df1["has_insurance_verify_clean"].value_counts())

# Cleaning "age" column
df1["age_clean"] = pd.to_numeric(
    df1["age"], 
    errors="coerce", 
)

broken_ages = df1.loc[(df1["age_clean"].isna()) & (df1["age"].notna())]
print(broken_ages)

df1["age"] = df1["age"].astype(str).str.replace("I", "1")

df1["age_clean"] = pd.to_numeric(
    df1["age"], 
    errors="coerce", 
)

broken_ages_2 = df1.loc[(df1["age_clean"].isna()) & (df1["age"].notna())]
print(broken_ages_2)

# No broken ages, the column is now clean
print(df1["age_clean"].isna().sum())
df1 = df1.dropna(subset=["age_clean"])
print(df1["age_clean"].isna().sum()) # Confirmed, no NaN cells in age

print(df1["admission_date"].head()) # To know the date formats

df1["admission_date_clean"] = pd.to_datetime(
    df1["admission_date"], 
    format="%d/%m/%Y", 
    errors="coerce", 
)

# Checking invalid admission dates
invalid_adm_dates = df1.loc[(df1["admission_date"].notna()) & (df1["admission_date_clean"].isna())]
print(invalid_adm_dates) # Some dates have O instead of 0, other I instead of 1

df1["admission_date_text"] = (df1["admission_date"]
                              .astype(str)
                              .str.replace("O", "0")
                              .str.replace("I", "1")
                             )
df1["admission_date_clean"] = pd.to_datetime(
    df1["admission_date_text"], 
    format="%d/%m/%Y", 
    errors="coerce", 
)
invalid_adm_dates_2 = df1.loc[(df1["admission_date_text"].notna()) & (df1["admission_date_clean"].isna())]
print(invalid_adm_dates_2) # Cleaned, invalid dates now

print(df1["discharge_date"].head()) # To know discharge dates format

df1["discharge_date_clean"] = pd.to_datetime(
    df1["discharge_date"], 
    format="%d/%m/%Y", 
    errors="coerce", 
)

invalid_discharge_dates = df1.loc[(df1["discharge_date_clean"].isna()) & (df1["discharge_date"].notna())]
print(invalid_discharge_dates) # There are invalid dates with O instead of 0

df1["discharge_date_text"] = df1["discharge_date"].astype(str).str.replace("O", "0")

df1["discharge_date_clean"] = pd.to_datetime(
    df1["discharge_date_text"], 
    format="%d/%m/%Y", 
    errors="coerce", 
)

# Confirming that there are no invalid dates
invalid_discharge_dates_2 = df1.loc[(df1["discharge_date_clean"].isna()) & (df1["discharge_date_text"].notna())]
print(invalid_discharge_dates_2) # No invalid dates

print(df1["discharge_date_clean"].isna().sum())

# Checking broken dates
broken_dates = df1["discharge_date_clean"] < df1["admission_date_clean"]
show_rows_broken_dates = df1.loc[broken_dates]
print(show_rows_broken_dates) # No broken dates

==========================================================================================
# df2 has broken dates, let's clean them
==========================================================================================
#Loading df2

df2 = pd.read_csv("broken_dates.csv", on_bad_lines="skip")
print(df2.info())

print(df2["Admission_Date"].head())

df2["Admission_Date_clean"] = pd.to_datetime(
    df2["Admission_Date"], 
    format="mixed", 
    errors="coerce", 
)

print(df2["Admission_Date_clean"].isna().sum())
print(df2.loc[(df2["Admission_Date_clean"].isna()) & (df2["Admission_Date"].notna())])
print(df2["Admission_Date_clean"].head())

admission_dates_updates = {
    "PID-4432": pd.Timestamp("2023-11-30"), 
    "PID-3321": pd.Timestamp("2023-02-28"), 
}

df2["Admission_Date_clean"] = (df2["Patient_ID"].map(admission_dates_updates)
                               .fillna(df2["Admission_Date_clean"])
                              )

print(df2["Admission_Date_clean"].isna().sum())
print(df2["Admission_Date"].isna().sum())

df2 = df2.dropna(subset="Admission_Date_clean")
print(df2["Admission_Date_clean"].isna().sum())

df2["Discharge_Date_clean"] = pd.to_datetime(
    df2["Discharge_Date"], 
    format="mixed", 
    errors="coerce", 
)

print(df2.loc[(df2["Discharge_Date_clean"].isna()) & (df2["Discharge_Date"].notna())])
print(df2["Discharge_Date_clean"].dtype)

brokenn_dates = df2["Discharge_Date_clean"] < df2["Admission_Date_clean"]
print(df2.loc[brokenn_dates])

df2.loc[brokenn_dates, ["Discharge_Date_clean", "Admission_Date_clean"]] = df2.loc[brokenn_dates, ["Admission_Date_clean", "Discharge_Date_clean"]].values

brokenn_dates_2 = df2["Discharge_Date_clean"] < df2["Admission_Date_clean"]
print(df2.loc[brokenn_dates_2]) # Clean, no broken dates

==============================================================================================
#Exporting df1
==============================================================================================

clean_columns={
    "patient_id": "patient_id", 
    "patient_name": "patient_name", 
    "age_clean": "age", 
    "medical_condition_clean": "medical_condition", 
    "has_insurance_verify_clean": "has_insurance_verify", 
    "admission_date_clean": "admission_date", 
    "discharge_date_clean": "discharge_date", 
}

my_series= pd.Series(index=df1.columns, data=df1.columns)

columns_data= my_series.map(clean_columns).dropna().index

clean_df= df1[columns_data]

clean_df=clean_df.rename(
    columns= clean_columns)


file_path= r"D:\Public health\my_cleaned_data.csv"

clean_df.to_csv(file_path, 
                index= False, 
                encoding= "utf-8-sig")
print("Export completed successfully")


    


   Unnamed: 0 PatientID#   PatientName   Age MedicalCondition AdmissionDate  \
0           0       P100      Patient_0  NaN                            NaN   
1           1       P101      Patient_1   29          Typhoid    04/06/2026   
2           2       P102      Patient_2   70          Malaria    14/05/2026   
3           3       P103      Patient_3  12I          Typhoid    17/02/2026   
4           4       P104      Patient_4   72          Typhoid    21/06/2026   

  DischargeDate HasInsurance_Verify  
0    29/01/2026               False  
1    15/06/2026                   N  
2    15/05/2026                 Yes  
3    27/02/2026                 Yes  
4    03/07/2026                TRUE  
Index(['Unnamed: 0', 'PatientID#', '  PatientName ', 'Age', 'MedicalCondition',
       'AdmissionDate', 'DischargeDate', 'HasInsurance_Verify'],
      dtype='object')
Index(['PatientID#', '  PatientName ', 'Age', 'MedicalCondition',
       'AdmissionDate', 'DischargeDate', 'HasInsurance_Verify'],